# Why are the 20250612 (GENE7) plates losing ~20% of embryos to QC?

Manual inspection says these plates are high quality, but `analysis_ready` passes only
74–96% of snips. Two candidate explanations:

1. **segmentation is failing** — masks are wrong, so downstream features are garbage; or
2. **QC flags are firing on good data** — masks are fine, thresholds are miscalibrated.

This notebook distinguishes them. The headline work product is a montage of every failed
embryo per plate, showing the extracted snip beside the raw focus-stacked frame with the
segmentation outline drawn on both, so mask quality is directly inspectable.

Everything is read from the pipeline's own artifacts. The surface-area band is re-derived
using the packaged reference curve and the product's real `k_upper`/`k_lower` (imported,
not copied), and `band_agrees` below asserts the reconstruction reproduces the flag the
pipeline actually wrote.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parents[2] if Path.cwd().name.startswith('202') else Path.cwd()
for p in (str(REPO_ROOT / 'src'), str(Path.cwd())):
    if p not in sys.path:
        sys.path.insert(0, p)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import qc_diagnostic_utils as Q

pd.set_option('display.width', 200)
FIGURES = Path.cwd() / 'figures'
FIGURES.mkdir(exist_ok=True)

def save(fig, name, dpi=110):
    for ext in ('png', 'pdf'):
        fig.savefig(FIGURES / f'{name}.{ext}', dpi=dpi, bbox_inches='tight')
    return fig

## 1. Load, and verify the reconstruction is faithful

`band_agrees` must be `True` for every snip. If it were not, nothing downstream could be
trusted, because the reconstructed band would not be the one that produced the flags.

In [ ]:
df = Q.add_surface_area_band(Q.load_analysis_ready())
print(f'snips: {len(df)}   plates: {df.experiment_id.nunique()}')
print(f'band reconstruction matches the pipeline flag on all snips: {bool(df.band_agrees.all())}')
print(f'  mismatches: {int((~df.band_agrees).sum())}')

from data_pipeline.quality_control.surface_area_qc.config import band_statement, resolve_config
print()
print(band_statement(resolve_config()))

## 2. Which flags actually cause the failures?

Flags are not mutually exclusive, so a snip can appear under several. `qc_fail_reasons`
gives the exact combination per failed snip.

In [ ]:
per_plate = (df.groupby('experiment_id')
               .agg(snips=('snip_id', 'size'), passed=('use_snip', 'sum'))
               .reindex(list(Q.EXPERIMENTS)))
per_plate['failed'] = per_plate.snips - per_plate.passed
per_plate['pass_%'] = (100 * per_plate.passed / per_plate.snips).round(1)
for spec in Q.FLAG_SPECS:
    counts = df.groupby('experiment_id')[spec.column].sum().reindex(list(Q.EXPERIMENTS))
    if counts.sum():
        per_plate[spec.column.replace('_flag', '')] = counts.astype(int)
per_plate

In [ ]:
print('exact failure combinations (all six plates pooled):')
print(df.loc[~df.use_snip, 'qc_fail_reasons'].value_counts().to_string())

n_fail = int((~df.use_snip).sum())
n_sa = int(df.loc[~df.use_snip, 'sa_outlier_flag'].sum())
print(f'\nsa_outlier_flag is implicated in {n_sa} of {n_fail} failures '
      f'({100 * n_sa / n_fail:.0f}%).')

In [ ]:
save(Q.plot_flag_counts(df), '01_flag_counts')
plt.show()

**Reading.** `sa_outlier_flag` dominates; every other flag is single digits. So the question
reduces to: is the surface-area check right to fire?

## 3. The surface-area failures track temperature, not stage

Each plate is a four-temperature block design (24 / 28.5 / 34 / 35 °C). If the check were
responding to genuine developmental progression, the failure rate should vary with stage.
It varies with **rearing temperature** instead.

In [ ]:
grid = (df.groupby(['start_age_hpf', 'temperature'])
          .agg(n=('snip_id', 'size'),
               sa_flagged=('sa_outlier_flag', 'sum'),
               too_small=('sa_too_small', 'sum'),
               too_large=('sa_too_large', 'sum'),
               median_area=('area_um2', 'median'),
               lower_cut=('sa_lower', 'first')))
grid['flag_%'] = (100 * grid.sa_flagged / grid.n).round(1)
grid[['median_area', 'lower_cut']] = (grid[['median_area', 'lower_cut']] / 1000).round(0)
grid

In [ ]:
save(Q.plot_flag_rate_by_temperature(df), '02_flag_rate_by_temperature')
plt.show()

**Reading.** At 24 °C the flag rate reaches 65% (30 hpf) and 56% (36 hpf); at 34 °C it is
~0–2%. Failures are almost entirely **too small**, and almost entirely in the cold cohort.

The cold cohort really is smaller — that is the point of the experiment. The question is
whether "smaller than a 28.5 °C wildtype reference" should be disqualifying.

## 4. Are these broken masks or healthy embryos just under a tight cut?

This is the discriminating measurement. A broken mask (e.g. yolk-only) lands *far* below the
cut; a healthy-but-small embryo lands *just* below it.

In [ ]:
save(Q.plot_marginality(df), '03_marginality_and_threshold_sensitivity')
plt.show()

In [ ]:
small = df.loc[df.sa_too_small].copy()
small['frac_of_cut'] = small.area_um2 / small.sa_lower
print('area as a fraction of the lower cut, for "too small" failures:')
print(small.frac_of_cut.describe().round(3).to_string())

bands = {'<50% of cut': (0, .5), '50-80%': (.5, .8), '80-100%': (.8, 1.01)}
print()
for label, (lo, hi) in bands.items():
    n = int(((small.frac_of_cut >= lo) & (small.frac_of_cut < hi)).sum())
    print(f'  {label:12s} {n:3d}')

In [ ]:
cfg = resolve_config()
other = [c for c in Q.FLAG_COLUMNS if c != 'sa_outlier_flag']
other_fail = df[other].any(axis=1)
rows = []
for k in [0.90, 0.85, 0.80, 0.75, 0.70, 0.60]:
    lower = (k / cfg.k_lower) * df.sa_lower
    sa = (df.area_um2 < lower) | (df.area_um2 > df.sa_upper)
    rows.append({'k_lower': k,
                 'sa_flagged': int(sa.sum()),
                 'pass_%': round(100 * (~(sa | other_fail)).sum() / len(df), 1)})
pd.DataFrame(rows).set_index('k_lower')

**Reading.** The overwhelming majority of "too small" failures sit within 20% of the cut,
median ~92% of it. That is the signature of a threshold that is too tight, not of broken
segmentation.

The sensitivity table shows the pass rate rising steeply as `k_lower` relaxes and then
flattening around 0.70–0.75, leaving a residue of ~16–23 flags. That residue is the set of
genuinely bad objects (see §6). The knee is the natural operating point.

Context from [`surface_area_qc/config.py`](../../../src/data_pipeline/quality_control/surface_area_qc/config.py):
`k_lower` was raised 0.7 → 0.8 → 0.9 on 2026-07-02 specifically to clear yolk-only SAM2
masks out of the passing band, with the documented cost of "also losing real thin/dorsal-pose
embryos". These plates are paying that cost at ~12 percentage points.

## 5. Would indexing on a temperature-corrected stage fix it?

`predicted_stage_hpf` is `start_age_hpf + elapsed_hours × rate(T)`. These are single-frame
snapshots, so `elapsed_hours = 0` and the predicted stage collapses to nominal clock time —
**temperature never enters**. The obvious fix is to index the band on the Arrhenius-corrected
developmental stage instead. Tested below; it does not work.

In [ ]:
print('Arrhenius-corrected stage, by nominal clock stage and rearing temperature:')
for nominal in (24.0, 30.0, 36.0):
    corrected = {t: round(Q.arrhenius_stage_hpf(nominal, t), 1) for t in Q.TEMPERATURES}
    print(f'  nominal {nominal:.0f} hpf -> {corrected}')

cf = Q.add_counterfactual_band(df)
recovered = int((cf.sa_outlier_flag & ~cf.cf_sa_outlier_flag).sum())
newly = int((~cf.sa_outlier_flag & cf.cf_sa_outlier_flag).sum())
print(f'\nrecovered by correcting the stage: {recovered}')
print(f'newly flagged by correcting the stage: {newly}')
print(f'net change in sa flags: {newly - recovered:+d}  '
      f'({int(df.sa_outlier_flag.sum())} -> {int(cf.cf_sa_outlier_flag.sum())})')
print()
print('where they move:')
print('  recovered:', (cf[cf.sa_outlier_flag & ~cf.cf_sa_outlier_flag]
                        .groupby(['start_age_hpf', 'temperature']).size().to_dict()))
print('  newly:    ', (cf[~cf.sa_outlier_flag & cf.cf_sa_outlier_flag]
                        .groupby(['start_age_hpf', 'temperature']).size().to_dict()))

In [ ]:
fig, summary = Q.plot_counterfactual(df)
save(fig, '04_counterfactual_arrhenius_stage')
plt.show()
summary[['total', 'as_run_%', 'corrected_%']].round(1)

In [ ]:
save(Q.plot_area_against_band(df), '05_area_against_reference_band')
plt.show()

**Reading.** Correcting the stage recovers the cold cohort but breaks the hot cohort, for a
net gain of ~10 snips. The reason is visible in the last figure: the X markers (median area
at the corrected stage) do not track the reference curve. Raising temperature advances
developmental stage without proportionally increasing projected area — total area is
dominated by a roughly fixed yolk + body mass set at fertilization.

So area is **not** a monotone proxy for stage across a temperature series, and no choice of
stage index makes a tight two-sided area band behave. This rules out the fix I expected to
work, and points at the width of the band rather than its indexing.

## 6. The genuinely bad objects

Not every failure is a false positive. The "too large" failures are real, and the flag is
right to catch them.

In [ ]:
big = df.loc[df.sa_too_large, ['experiment_id', 'well_index', 'temperature', 'area_um2',
                               'sa_upper', 'length_um', 'width_um', 'focus_flag']].copy()
big['x_over_cut'] = (big.area_um2 / big.sa_upper).round(2)
big['area_k'] = (big.area_um2 / 1000).round(0)
print(f'all {len(big)} "too large" failures also flagged out-of-focus: '
      f'{bool(big.focus_flag.all())}')
big.drop(columns=['area_um2', 'sa_upper']).reset_index(drop=True)

**Reading.** These have `length_um ≈ 4400` and `width_um ≈ 2000` — essentially the full field
of view — at 5–7× the upper cut. They are whole-frame masks: wells where the embryo was not
found and SAM2 segmented the background. Every one is independently flagged out of focus.
This is the small, real segmentation-failure population, and it is exactly what the upper cut
and the focus check exist to remove.

## 7. Montages — every failed embryo, per plate

The main work product. Per cell: **left** = extracted snip (what the model and QC see, at the
fixed µm/px snip scale), **right** = raw focus-stacked frame cropped around the mask. The cyan
contour is the segmentation mask, drawn on both so a bad mask is obvious against the raw
image. Nested rectangles encode which QC flags fired — count the rings, read the colors off
the legend; the outermost ring is the first flag in canonical order.

Also written to `figures/montage_<plate>.png` at higher resolution for zooming.

In [ ]:
montages = {}
for experiment_id in Q.EXPERIMENTS:
    fig = Q.build_failure_montage(experiment_id, df)
    if fig is None:
        print(f'{experiment_id}: no failures')
        continue
    save(fig, f'montage_{experiment_id}', dpi=150)
    montages[experiment_id] = fig
    n_fail = int((~df.loc[df.experiment_id == experiment_id, 'use_snip']).sum())
    print(f'{experiment_id}: {n_fail} failed embryos rendered')

In [ ]:
montages['20250612_30hpf_ctrl_atf6']

In [ ]:
montages['20250612_36hpf_wfs1_ctcf']

In [ ]:
montages['20250612_24hpf_ctrl_atf6']

In [ ]:
for experiment_id in ('20250612_24hpf_wfs1_ctcf', '20250612_30hpf_wfs1_ctcf',
                     '20250612_36hpf_ctrl_atf6'):
    display(montages[experiment_id])

## 8. Preliminary conclusions

**Segmentation is not the problem.** Across all six montages the cyan contour tracks the
embryo boundary on the raw frame. The flagged embryos are, with few exceptions, well-segmented
and normal-looking — consistent with your manual inspection.

**The loss is one check, firing on good data.** `sa_outlier_flag` is implicated in 96 of 106
failures. 87 of those are "too small", and 75 of the 87 sit within 20% of the cut (median
~92% of it). That is a tight-threshold signature, not a broken-mask signature.

**The mechanism is a reference-population mismatch, not a bug.** `k_lower = 0.9 × p5` is
evaluated against a wildtype curve built at ~28.5 °C. Cold-reared embryos are genuinely and
correctly smaller, so a healthy 24 °C cohort lands just under the 5th percentile of a
reference it does not belong to. This is worst exactly where the biology is most interesting:
65% of 30 hpf / 24 °C snips are discarded.

**The obvious fix does not work.** Re-indexing the band on Arrhenius-corrected developmental
stage recovers 55 cold-cohort snips but newly flags 45 hot-cohort ones — net 10. Temperature
changes stage and projected area non-proportionally, so no stage index rescues a tight
two-sided area band on a temperature series.

**What does move the needle.** Relaxing `k_lower`: 0.90 → 0.80 takes the pass rate from 81.9%
to 91.5%; → 0.75 gives 93.9%, after which the curve flattens with ~16–23 residual flags —
essentially the 9 whole-frame blowouts plus a few genuinely tiny objects. The knee is around
0.75, and `k_lower` was 0.7 before being raised on 2026-07-02.

**Caveats.** The raised `k_lower` exists for a documented reason — separating yolk-only masks
from real small embryos, which area alone cannot do
(`surface_area_qc_pose_confound.md`). Relaxing it globally would re-admit whatever
population motivated the change, which is not visible in these six plates. Two narrower
options avoid that: make the reference temperature-aware (fit p5/p95 per rearing temperature
rather than pooling), or make the tolerance per-experiment-class so temperature series get a
wider lower band while standard plates keep 0.9. Both are product decisions, not something
this notebook should settle.

**Also worth noting:** `predicted_stage_hpf` silently ignores temperature for any
single-frame experiment, because the rate multiplies an elapsed time of zero. That is correct
arithmetic and misleading as a column name — anything downstream treating it as a
developmental stage for a snapshot plate is really reading nominal clock time. It affects
more than this QC check.